# taskb_final

Common E*=8, fresh heads and twelve official-val runs. See [Task B guide](README_taskb.md) for report evidence, execution order and historical-source limitations.


In [ ]:
# Run before any numerical imports. Colab supplies torch/CUDA; do not replace that stack.
import sys,subprocess,importlib.util,importlib.metadata,json
BOOTSTRAP_REPORT={'python':sys.version,'packages':{},'installs_performed':False}
for name in ('torch','numpy','pandas'):
    if importlib.util.find_spec(name) is None:raise RuntimeError('Colab base dependency missing: '+name+'. Use a standard Colab Python runtime; no automatic torch/CUDA replacement is performed.')
    BOOTSTRAP_REPORT['packages'][name]=importlib.metadata.version(name)
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
BOOTSTRAP_REPORT['pip_check']={'returncode':check.returncode,'output':check.stdout+check.stderr}
relevant=[line for line in (check.stdout+check.stderr).splitlines() if line.lower().startswith(('torch ','numpy ','pandas ')) or any((' requires '+p+' ') in line.lower() or (' requires '+p+',') in line.lower() for p in ('torch','numpy','pandas'))]
if relevant:raise RuntimeError('Dependency conflicts relevant to head training:\n'+'\n'.join(relevant))
print('Bootstrap:',BOOTSTRAP_REPORT['packages'],'| pip check returncode:',check.returncode)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


`baseline_campaign` là nguồn chỉ đọc từ notebook 1 cũ; không train lại development baseline. Chỉ chấp nhận implementation v1.0.0 có hash đã xác minh, kèm kiểm tra đầy đủ contract/checkpoint/history. Final lock chờ đủ mọi seed và metric ở epoch chung. Checkpoint DVR/final cũ không tự migration/resume: tiếp tục bằng notebook gốc hoặc dùng campaign tối ưu mới.


In [ ]:
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103',
 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/taskb_training/FashionIQ',
 'campaign_id': 'heads_v1_20261103_ioopt_v1',
 'local_feature_cache': '/content/taskb_feature_memmap',
 'phase': 'final',
 'models_to_run': ['xaa_noft', 'xaa_mr', 'xaa_sr', 'xaa_full_dvr'],
 'seeds_to_run': None,
 'eval_batch_size': 256,
 'resume': True,
 'baseline_campaign': '/content/drive/MyDrive/ĐATN/data/processed/taskb_training/FashionIQ/heads_v1_20261103',
 'profile_batches': 2}

ARCHITECTURE_SOURCES = {'ModifierGuidedRefinement': {'cell_index': 17,
                              'class_source_sha256': '3af4f48aec472e41b87a84c4642ea9ac1151f140737d0233f1b25db9a55d9c4f',
                              'notebook_sha256': 'f8b584942a6d3aaf7ac0c485b84043df71000dc6a194488f8cf5831693f15daa'},
 'SelfGuidedRefinement': {'cell_index': 19,
                          'class_source_sha256': 'd5c4c475dd4876b34dca889e464ed3f6f709cfae52d5ff7fd0ef76260edeef1c',
                          'notebook_sha256': 'f8b584942a6d3aaf7ac0c485b84043df71000dc6a194488f8cf5831693f15daa'},
 'XAADVR': {'cell_index': 21,
            'class_source_sha256': '13d61b06383efd13f27b536b2fe5184708aa431827731ee4aa3dae68e2f484af',
            'notebook_sha256': 'f8b584942a6d3aaf7ac0c485b84043df71000dc6a194488f8cf5831693f15daa'},
 'XAAFusionModule': {'cell_index': 15,
                     'class_source_sha256': 'e8fd37e457a80f88f5812f36ff1bf4f8effeecdd7e3ba7d587ee07c7aae86659',
                     'notebook_sha256': 'f8b584942a6d3aaf7ac0c485b84043df71000dc6a194488f8cf5831693f15daa'}}
LOSS_SOURCE_RECORD = {'cell_index': 17,
 'code_sha256': 'e56eaa05bcff53a84424cbf858d5f325ad71de9c9726cc4768999df689c12857',
 'notebook_sha256': '64b2e0bcda09710315e30b20ebc4169ef6f5034d28398fc8b9c83c510e1958f8',
 'source_notebook': '[ĐATN][TaskB]XAA_Baseline_NoFT.ipynb'}
EXPECTED_EPOCH_RULE = ('At common evaluated epochs, maximize the mean development Macro Mean across all four variants and configured '
 'seeds; choose earliest tie. Freeze one common E* and chosen hyperparameters before any final training or '
 'official-val evaluation.')
PACKAGE_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95',
 'feature_audit.json': 'a4ceb9b4352762891315eb2f3f92d5e351631546292a9a1ac7e22b589693581b',
 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342',
 'preparation_status.json': '0aaab47d2d1ec6222962ec65ba27d38a70b2996e1b26be1ec182aa139230d5fa',
 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb',
 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21'}
SOURCE_RECORDS = [{'path': '/Users/tuankhong/Documents/VisFit/notebooks/[ĐATN][TaskB]XAA_Baseline_NoFT.ipynb',
  'sha256': '64b2e0bcda09710315e30b20ebc4169ef6f5034d28398fc8b9c83c510e1958f8'},
 {'path': '/Users/tuankhong/Documents/VisFit/notebooks/[ĐATN][TaskB]XAA_DVR.ipynb',
  'sha256': 'f8b584942a6d3aaf7ac0c485b84043df71000dc6a194488f8cf5831693f15daa'},
 {'path': '/Users/tuankhong/Downloads/[ĐATN][TaskB]FashionIQ_Benchmark_Preparation.ipynb',
  'sha256': '05096767e23561c7a1f87c4fef1c00bdff36bfd3324bf18c50de9a75ea7d0ab7'},
 {'path': '/Users/tuankhong/Documents/VisFit/README.md',
  'sha256': '81fe5b0133589962d3292d657726c7a0a77214b393db51e1d105fa75fae1cc55'},
 {'path': '/Users/tuankhong/Documents/VisFit/docs/visfit_weekly_log.md',
  'sha256': 'ed2b856966824ab6ecf0481ea935c6179497b9d737675d5cfd9c99ce8579ae27'}]
SHARED_LOGIC_SHA256 = '17cd718acb556744f5beff29d72a8f21562f25f09aecd34ef745ff760d48896c'
EVALUATOR_SOURCE_SHA256 = '5ce0a6897a886b8ddd9cac139cdf6a13f27ac8baaea18c89aa0f0746cf548848'
MODEL_LOSS_SOURCE_SHA256 = '572c80f35d2f30e5ea42c4c588d7bbbc5122a06badd1aa66bb175be031f27b09'


In [ ]:
# Shared logic: identical bytes/version in all three notebooks. Feature-only training; no encoder imports.
import os,sys,json,math,time,copy,random,hashlib,uuid,re,traceback,platform,tempfile,subprocess,importlib.metadata
from pathlib import Path
from collections import OrderedDict,defaultdict
from contextlib import contextmanager
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader

CATEGORIES=('dress','shirt','toptee');KS=(1,5,10,50)
LOGIC_VERSION='visfit-taskb-heads-v1.1.0-ioopt'
class IntegrityError(ValueError):pass
class PlanError(ValueError):pass

def require(condition,message,kind=IntegrityError):
    if not condition:raise kind(message)
def canonical(x):return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode()
def fingerprint(x):return hashlib.sha256(canonical(x)).hexdigest()
def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(p):return json.loads(Path(p).read_text())
def read_csv(p):return pd.read_csv(p,dtype=str,keep_default_na=False)
def atomic_bytes(p,raw,immutable=False):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    if immutable and p.exists():require(p.read_bytes()==raw,'Existing artifact differs; use a new campaign: '+str(p));return
    stage=p.with_name(p.name+'.'+uuid.uuid4().hex+'.tmp')
    with stage.open('wb') as f:f.write(raw);f.flush()
    os.replace(stage,p)
def write_json(p,x,immutable=False):atomic_bytes(p,json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False).encode(),immutable)
def write_csv(p,x,immutable=False):atomic_bytes(p,x.to_csv(index=False,lineterminator='\n').encode(),immutable)
def atomic_torch_save(p,state):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+'.'+uuid.uuid4().hex+'.tmp')
    with tmp.open('wb') as f:torch.save(state,f);f.flush()
    os.replace(tmp,p)
def safe_checkpoint(p):return torch.load(p,map_location='cpu',weights_only=True)
def safe_child(root,relative):
    root=Path(root).resolve();p=(root/relative).resolve();require(root in p.parents,'Artifact path escapes package: '+str(relative));return p

def redact(text):
    text=str(text)
    for key,value in os.environ.items():
        if len(value)>=6 and re.search(r'TOKEN|PASSWORD|SECRET|API_KEY|ACCESS_KEY',key,re.I):text=text.replace(value,'<redacted>')
    return re.sub(r'(?i)([?&](?:token|key|sig|signature|access_token)=)[^\s&]+',r'\1<redacted>',text)

def runtime_contract():
    return {'python':platform.python_version(),'torch':str(torch.__version__),'numpy':np.__version__,'pandas':pd.__version__,'dtype':'float32','amp':False,'matmul_precision':'highest','tf32':False,'num_workers':0}
def execution_environment():
    return {**runtime_contract(),'platform':platform.platform(),'device':str(DEVICE),'cuda':torch.version.cuda,'device_name':torch.cuda.get_device_name(DEVICE) if DEVICE.type=='cuda' else platform.machine(),
      'reproducibility':'Epoch-boundary resume, worker count 0, saved Python/NumPy/Torch/CUDA/DataLoader RNG; no cross-device/GPU bit-exact promise.'}
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def capture_rng(generator=None):
    state=np.random.get_state()
    return {'python':random.getstate(),'numpy':[state[0],torch.tensor(state[1].astype(np.int64)),int(state[2]),int(state[3]),float(state[4])],
      'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],'loader':generator.get_state() if generator is not None else None}
def restore_rng(state,generator=None):
    random.setstate(state['python']);n=state['numpy'];np.random.set_state((n[0],n[1].cpu().numpy().astype(np.uint32),n[2],n[3],n[4]));torch.set_rng_state(state['torch'])
    if state['cuda']:
        require(torch.cuda.is_available() and len(state['cuda'])==torch.cuda.device_count(),'CUDA RNG device count changed; resume refused.');torch.cuda.set_rng_state_all(state['cuda'])
    if generator is not None and state['loader'] is not None:generator.set_state(state['loader'])
@contextmanager
def preserve_rng():
    state=capture_rng()
    try:yield
    finally:restore_rng(state)
def cpu_clone(x):
    if isinstance(x,torch.Tensor):return x.detach().cpu().clone()
    if isinstance(x,dict):return {k:cpu_clone(v) for k,v in x.items()}
    if isinstance(x,list):return [cpu_clone(v) for v in x]
    if isinstance(x,tuple):return tuple(cpu_clone(v) for v in x)
    return copy.deepcopy(x)
def model_hash(model,exclude=()):
    h=hashlib.sha256()
    for name,value in sorted(model.state_dict().items()):
        if any(name.startswith(p) for p in exclude):continue
        h.update(name.encode());h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
def normalized(a):
    a=np.asarray(a,dtype=np.float32);require(a.ndim==2 and np.isfinite(a).all(),'Embedding matrix not finite/2-D.');n=np.linalg.norm(a,axis=1,keepdims=True);require((n>1e-8).all(),'Zero embedding.');return np.ascontiguousarray(a/n)

def validate_training_plan(plan):
    required=['schema_version','models','seeds','architecture','initial_hyperparameters','loss','development','final_training']
    require(all(k in plan for k in required),'training_plan missing required keys.',PlanError)
    require(plan['schema_version']=='training-plan-v1','Unsupported training_plan schema.',PlanError)
    expected={'xaa_noft':(False,False),'xaa_mr':(True,False),'xaa_sr':(False,True),'xaa_full_dvr':(True,True)}
    require(set(plan['models'])==set(expected),'Plan must contain exactly the four No-FT/DVR configurations.',PlanError)
    for name,flags in expected.items():require((plan['models'][name]['use_mr'],plan['models'][name]['use_sr'])==flags,'MR/SR flags conflict: '+name,PlanError)
    seeds=plan['seeds'];require(isinstance(seeds,list) and seeds and all(type(i) is int for i in seeds) and len(set(seeds))==len(seeds),'Invalid seeds.',PlanError)
    a=plan['architecture'];spec={'text_dim':512,'reference_dim':768,'embed_dim':512,'num_heads':8,'attention_dropout':0.1,'dvr_hidden':256,'dvr_alpha':1.0,'mr_then_sr':True}
    require(all(a.get(k)==v for k,v in spec.items()),'Architecture conflicts with source; no implicit architectural change.',PlanError)
    require(a['source_sha256']==ARCHITECTURE_SOURCES,'Architecture source version mismatch.',PlanError)
    h=plan['initial_hyperparameters']
    keys=['optimizer','learning_rate','weight_decay','betas','eps','train_batch_size','max_development_epochs','eval_every_epochs','patience_evaluations','gradient_clip_norm','scheduler','warmup_ratio','drop_last_train','logit_scale_init','logit_scale_clamp']
    require(all(k in h for k in keys),'Missing training hyperparameter(s): '+str([k for k in keys if k not in h]),PlanError)
    require(h['optimizer']=='AdamW' and h['scheduler']=='5% linear warmup then cosine decay','Unsupported optimizer/scheduler; review training_plan.',PlanError)
    for k in ('train_batch_size','max_development_epochs','eval_every_epochs','patience_evaluations'):require(type(h[k]) is int and h[k]>0,'Invalid '+k,PlanError)
    require(h['max_development_epochs']>=h['eval_every_epochs'],'No development evaluation within budget.',PlanError)
    require(h['learning_rate']>0 and h['weight_decay']>=0 and h['eps']>0 and h['gradient_clip_norm']>0 and 0<=h['warmup_ratio']<1,'Invalid optimizer values.',PlanError)
    require(len(h['betas'])==2 and all(0<=v<1 for v in h['betas']),'Invalid AdamW betas.',PlanError)
    require(h['drop_last_train'] is True and h['logit_scale_init']=='ln(1/0.07)' and np.allclose(h['logit_scale_clamp'],[0,math.log(100)],atol=1e-12),'Loss/sampling/logit policy conflicts.',PlanError)
    require(plan['loss']['name']=='symmetric multi-positive InfoNCE' and plan['loss']['auxiliary_losses'] is False and plan['loss']['source']==LOSS_SOURCE_RECORD,'Loss source/schema conflict.',PlanError)
    require(plan['development']['checkpoint_metric']=='Macro Mean = mean_category((R@10_category + R@50_category)/2), percent','Development metric conflict.',PlanError)
    final=plan['final_training'];require(final['reinitialize_head'] and not final['reuse_old_checkpoint'] and final['frozen_encoder'] and not final['official_val_used_for_selection'],'Final training policy conflicts.',PlanError)
    require(final['epoch_lock_rule']==EXPECTED_EPOCH_RULE,'Epoch rule changed; review plan rather than silently reinterpreting it.',PlanError)
    return h


In [ ]:
DVR_HIDDEN=256
DVR_ALPHA=1.0

class XAAFusionModule(nn.Module):
    def __init__(self, text_dim=512, img_dim=768, embed_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.ref_proj = nn.Linear(img_dim, embed_dim)
        self.ref_ln   = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                          batch_first=True, dropout=dropout)
        self.out_ln = nn.LayerNorm(embed_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def forward(self, text_hidden, text_mask, reference_hidden):
        ref = self.ref_ln(self.ref_proj(reference_hidden))              # [B,50,512]
        att, _ = self.attn(query=text_hidden, key=ref, value=ref)       # [B,77,512]
        x = self.out_ln(text_hidden + att)                              # residual + LN
        m = text_mask.unsqueeze(-1)                                     # [B,77,1]
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1.0)               # masked mean
        return F.normalize(pooled, dim=-1)

class ModifierGuidedRefinement(nn.Module):
    def __init__(self, dim=512, hidden=DVR_HIDDEN, alpha=DVR_ALPHA):
        super().__init__()
        self.alpha = float(alpha)
        self.v_proj = nn.Linear(dim, hidden)       # W_v
        self.t_proj = nn.Linear(dim, hidden)       # W_t
        self.act = nn.GELU()
        self.out = nn.Linear(hidden, 1)            # gate head
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)   # identity tại khởi tạo

    def forward(self, v, text_hidden, text_mask):
        # v: [B,N,D] | text_hidden: [B,L,D] | text_mask: [B,L] (1 = token thật)
        m = text_mask.unsqueeze(-1).to(text_hidden.dtype)
        t = (text_hidden * m).sum(1) / m.sum(1).clamp(min=1.0)                  # masked mean, bỏ padding   [B,D]
        h = self.act(self.v_proj(v) + self.t_proj(t).unsqueeze(1))              # [B,N,hidden]
        gate = 1.0 + self.alpha * torch.tanh(self.out(h).squeeze(-1))          # [B,N]  ∈ (1-α, 1+α)
        return v * gate.unsqueeze(-1), gate

class SelfGuidedRefinement(nn.Module):
    def __init__(self, dim=512, hidden=DVR_HIDDEN, alpha=DVR_ALPHA):
        super().__init__()
        self.alpha = float(alpha)
        self.v_proj = nn.Linear(dim, hidden)       # W_v
        self.c_proj = nn.Linear(dim, hidden)       # W_c (ngữ cảnh thị giác toàn ảnh)
        self.act = nn.GELU()
        self.out = nn.Linear(hidden, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, v):                           # CHỈ nhận image tokens — không có đường vào cho text
        ctx = v.mean(dim=1)                                                     # [B,D]
        h = self.act(self.v_proj(v) + self.c_proj(ctx).unsqueeze(1))            # [B,N,hidden]
        gate = 1.0 + self.alpha * torch.tanh(self.out(h).squeeze(-1))          # [B,N]
        return v * gate.unsqueeze(-1), gate

class XAADVR(nn.Module):
    def __init__(self, use_mr=False, use_sr=False, text_dim=512, img_dim=768, embed_dim=512, num_heads=8,
                 dropout=0.1, dvr_hidden=DVR_HIDDEN, dvr_alpha=DVR_ALPHA):
        super().__init__()
        self.use_mr, self.use_sr = bool(use_mr), bool(use_sr)
        # --- XAA cơ bản: cùng thứ tự khởi tạo với XAAFusionModule ---
        self.ref_proj = nn.Linear(img_dim, embed_dim)
        self.ref_ln   = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True, dropout=dropout)
        self.out_ln = nn.LayerNorm(embed_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
        # --- DVR: tạo sau, rồi trả RNG về trạng thái ngay sau khi tạo XAA ---
        _rng = torch.get_rng_state()
        self.mr = ModifierGuidedRefinement(embed_dim, dvr_hidden, dvr_alpha) if self.use_mr else None
        self.sr = SelfGuidedRefinement(embed_dim, dvr_hidden, dvr_alpha) if self.use_sr else None
        torch.set_rng_state(_rng)

    def forward(self, text_hidden, text_mask, reference_hidden, return_gates=False):
        ref = self.ref_ln(self.ref_proj(reference_hidden))              # Image Projection  [B,50,512]
        gates = {}
        if self.mr is not None:
            ref, gates["mr"] = self.mr(ref, text_hidden, text_mask)    # MR
        if self.sr is not None:
            ref, gates["sr"] = self.sr(ref)                            # SR (sau MR)
        att, _ = self.attn(query=text_hidden, key=ref, value=ref)       # XAA không đổi
        x = self.out_ln(text_hidden + att)
        m = text_mask.unsqueeze(-1)
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1.0)
        out = F.normalize(pooled, dim=-1)
        return (out, gates) if return_gates else out

def multi_positive_infonce(comp, tgt, target_idx, logit_scale):
    comp = F.normalize(comp.float(), dim=-1)
    tgt  = F.normalize(tgt.float(),  dim=-1)
    logits = comp @ tgt.T * logit_scale.exp()                           # [B,B]
    pos = (target_idx[:, None] == target_idx[None, :]).float()          # positive_mask[i,j]
    def one_way(lg, p):
        logp = F.log_softmax(lg, dim=1)
        return (-(logp * p).sum(1) / p.sum(1).clamp(min=1.0)).mean()
    return 0.5 * (one_way(logits, pos) + one_way(logits.T, pos.T))

def build_model(plan,name):
    a=plan['architecture'];spec=plan['models'][name]
    args=dict(text_dim=a['text_dim'],img_dim=a['reference_dim'],embed_dim=a['embed_dim'],num_heads=a['num_heads'],dropout=a['attention_dropout'])
    if name=='xaa_noft':return XAAFusionModule(**args)
    return XAADVR(**args,use_mr=spec['use_mr'],use_sr=spec['use_sr'],dvr_hidden=a['dvr_hidden'],dvr_alpha=a['dvr_alpha'])
def query_forward(model,reference_hidden,text_hidden,text_mask):
    return model(text_hidden,text_mask,reference_hidden)


In [ ]:
class FeatureReader:
    def __init__(self,package_root,feature_root,kind,key,index_sha,cache_root):
        self.root=safe_child(feature_root,kind);self.kind=kind;self.key=key;index_path=self.root/'index.json'
        require(file_hash(index_path)==index_sha,'Feature index checksum mismatch: '+kind)
        meta=read_json(index_path);require(meta['fingerprint']==key and meta['kind']==kind,'Feature index fingerprint/kind mismatch.')
        self.ids=meta['ids'];self.id2idx={v:i for i,v in enumerate(self.ids)};require(len(self.ids)==len(self.id2idx),'Duplicate feature IDs.')
        self.tail=tuple(meta['shape_tail']);self.dtype=np.dtype(meta['dtype']);self.step=meta['shard_size'];require(type(self.step) is int and self.step>0,'Invalid shard_size.')
        expected={'text':((77,512),'float32'),'mask':((77,),'int64'),'reference':((50,768),'float32'),'target':((512,),'float32')}
        require((self.tail,str(self.dtype))==expected[kind],'Feature schema incompatible with source head: '+kind)
        mapping=read_json(safe_child(feature_root,kind+'_id2idx.json'));require(mapping==self.id2idx,'Mapping differs from index ID order: '+kind)
        self.cache=Path(cache_root)/key/kind;self.cache.mkdir(parents=True,exist_ok=True);self.loaded=OrderedDict();self.validated={};self.schemas={};self.sealed=False;self.io_stats=defaultdict(float)
    def schema(self,start):
        if start in self.schemas:return self.schemas[start]
        require(not self.sealed,'Unverified shard metadata during training; rerun full preflight.')
        self.io_stats['drive_metadata_reads']+=1
        end=min(start+self.step,len(self.ids));name=f'{start:07d}_{end:07d}'
        marker=self.root/(name+'.json');require(marker.is_file(),'Missing committed feature shard marker: '+str(marker));meta=read_json(marker)
        expected={'fingerprint':self.key,'kind':self.kind,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.tail],'dtype':str(self.dtype)}
        require(all(meta.get(k)==v for k,v in expected.items()),'Shard fingerprint/ID/schema mismatch: '+str(marker))
        result=(self.root/(name+'.npy'),meta);self.schemas[start]=result;return result
    def begin_preflight(self):
        # Revalidate Drive source and every cached NPY on each startup/resume, even in the same session.
        self.sealed=False;self.schemas.clear();self.validated.clear();self.loaded.clear();self.io_stats=defaultdict(float)
    def local_signature(self,path):
        try:s=path.stat()
        except FileNotFoundError:raise IntegrityError('Local feature cache missing; stop and rerun package/preflight: '+str(path))
        return (s.st_dev,s.st_ino,s.st_size,s.st_mtime_ns,s.st_ctime_ns)
    def local_shard(self,start):
        source,meta=self.schema(start);local=self.cache/source.name
        if start not in self.validated:
            require(not self.sealed,'Local cache is not verified; rerun package/preflight.')
            require(source.is_file(),'Missing source feature NPY; never regenerate features: '+str(source))
            clock=time.perf_counter();require(file_hash(source)==meta['sha256'],'Source feature checksum mismatch: '+str(source))
            self.io_stats['source_checksum_seconds']+=time.perf_counter()-clock;self.io_stats['source_shards_verified']+=1
            if not local.is_file() or file_hash(local)!=meta['sha256']:
                clock=time.perf_counter();tmp=local.with_name(local.name+'.'+uuid.uuid4().hex+'.tmp')
                with source.open('rb') as src,tmp.open('wb') as dst:
                    for block in iter(lambda:src.read(8*1024*1024),b''):dst.write(block)
                require(file_hash(tmp)==meta['sha256'],'Local copy checksum mismatch.');os.replace(tmp,local)
                self.io_stats['copy_seconds']+=time.perf_counter()-clock;self.io_stats['copied_bytes']+=local.stat().st_size
            a=np.load(local,mmap_mode='r',allow_pickle=False);require(list(a.shape)==meta['shape'] and a.dtype==self.dtype,'NPY header mismatch.')
            self.validated[start]=self.local_signature(local)
        # ONLY /content/local operations below. Never stat/hash/read a Drive path in the hot path.
        require(self.local_signature(local)==self.validated[start],'Local feature cache changed; stop and rerun full preflight to repair/revalidate: '+str(local))
        if start not in self.loaded:
            if len(self.loaded)>=16:self.loaded.popitem(last=False)
            self.loaded[start]=np.load(local,mmap_mode='r',allow_pickle=False)
        self.loaded.move_to_end(start);return self.loaded[start]
    def take_indices(self,indices):
        indices=np.asarray(indices,dtype=np.int64);require(indices.ndim==1 and len(indices)>0 and (indices>=0).all() and (indices<len(self.ids)).all(),'Feature row index out of bounds.')
        output=np.empty((len(indices),*self.tail),dtype=self.dtype)
        for start in np.unique(indices//self.step*self.step):
            positions=np.flatnonzero(indices//self.step*self.step==start);a=self.local_shard(int(start));output[positions]=a[indices[positions]-start]
        return output
    def take_ids(self,ids):return self.take_indices([self.id2idx[v] for v in ids])
    def audit_all(self):
        self.begin_preflight()
        for start in range(0,len(self.ids),self.step):
            a=self.local_shard(start)
            require(np.isfinite(a).all(),'NaN/Inf feature: '+self.kind)
            if self.kind=='mask':require(np.isin(a,[0,1]).all() and (a.sum(1)>0).all(),'Invalid attention mask.')
            else:
                norms=np.linalg.norm(np.asarray(a).reshape(len(a),-1).astype(np.float64),axis=1);require((norms>1e-8).all(),'Zero feature sample.')
                if self.kind=='target':require(np.allclose(norms,1,atol=1e-5,rtol=1e-5),'Target not L2 normalized.')
        self.sealed=True
        return {'kind':self.kind,'count':len(self.ids),'shards':math.ceil(len(self.ids)/self.step),'all_checksums_and_values_checked':True,'metadata_cached_in_RAM':True,'io':dict(self.io_stats)}

def parse_frame(path):
    frame=read_csv(path)
    for col in ('source_row','gallery_index','text_feature_index','reference_feature_index','target_feature_index'):
        if col in frame:
            require(frame[col].str.fullmatch(r'\d+').all(),'Invalid integer column '+col);frame[col]=frame[col].astype(np.int64)
    return frame

class PreparedPackage:
    def __init__(self,root,cache_root,anchors,full_check=True,fixture=False):
        self.root=Path(root).resolve();require(self.root.is_dir(),'Preparation directory missing: '+str(self.root));self.full_verified=False
        require(anchors or fixture,'An observed preparation anchor set is required.')
        for rel,sha in anchors.items():require(file_hash(safe_child(self.root,rel))==sha,'Preparation changed after authoring: '+rel)
        self.status=read_json(self.root/'preparation_status.json');self.plan=read_json(self.root/'training_plan.json');self.protocol=read_json(self.root/'protocol_config.json');self.meta=read_json(self.root/'feature_metadata.json');self.audit=read_json(self.root/'feature_audit.json')
        self.data_audit=read_json(self.root/'data_audit.json')
        require(self.data_audit['passed'] and not self.data_audit['partial_mode'] and self.data_audit['schema_error_count']==0 and self.data_audit['both_empty_captions_count']==0 and self.data_audit['missing_or_corrupt_image_count']==0 and self.data_audit['required_images_all_decoded'],'Preparation data audit failed/incomplete; no partial mode.')
        require(all(self.status.get(k) is True for k in ('protocol_locked','training_plan_spec_locked','features_complete','tests_passed','ready_for_development_training')),'Preparation is not ready for development training.')
        self.h=validate_training_plan(self.plan);self.key=self.meta['fingerprint'];require(bool(re.fullmatch(r'[0-9a-f]{64}',self.key)),'Invalid feature fingerprint.')
        require(self.status['feature_fingerprint']==self.key==self.audit['fingerprint'] and self.audit['passed'] and not self.audit['partial'],'Feature status/audit mismatch.')
        require(self.meta['status']=='complete' and self.meta['legacy_arrays_mixed'] is False and self.audit['historical_features_reused'] is False,'Historical/incomplete feature package refused.')
        require(self.protocol['protocol']=='FashionIQ-OfficialVal-v1' and self.protocol['reference_policy']=='retain in gallery (FashionIQ)','Protocol/reference policy mismatch.')
        require(self.protocol['metrics_planned']['recall_at']==list(KS) and self.protocol['metrics_planned']['unit']=='percent','Metric configuration mismatch.')
        require(self.meta['contract']['caption_policy']==self.protocol['caption_policy'] and self.meta['contract']['dtype']=='float32','Caption/precision contract mismatch.')
        self.feature_root=safe_child(self.root,'features/'+self.key)
        require(file_hash(self.root/'feature_audit.json')==self.meta['feature_audit_sha256'],'Feature audit checksum mismatch.')
        self.stores={kind:FeatureReader(self.root,self.feature_root,kind,self.key,self.meta['store_index_sha256'][kind],cache_root) for kind in ('text','mask','reference','target')}
        require(self.stores['text'].ids==self.stores['mask'].ids,'Text/mask ID order mismatch.')
        for kind in ('text','reference','target'):require(len(self.stores[kind].ids)==self.meta['counts'][kind],'Feature count mismatch: '+kind)
        require(fingerprint(self.stores['text'].ids)==self.meta['query_order_sha256'],'Text order hash mismatch.')
        for kind in ('reference','target'):require(fingerprint(self.stores[kind].ids)==self.meta[kind+'_order_sha256'],'Image ID order hash mismatch.')
        self.frames={};self.manifest_hashes={}
        names=('official_train_queries','official_val_queries','official_val_gallery','development_train','development_val','development_gallery')
        require(set(self.meta['feature_views'])==set(names),'Feature view roles changed/missing.')
        for name in names:
            record=self.meta['feature_views'][name];require(record['fingerprint']==self.key,'View fingerprint mismatch.')
            path=safe_child(self.root,record['path']);require(file_hash(path)==record['sha256'],'Feature view checksum mismatch: '+name)
            frame=parse_frame(path);base_path=self.root/(name+'.csv');base=parse_frame(base_path)
            require(len(frame)==record['rows'] and set(base.columns)<=set(frame.columns),'Manifest count/schema mismatch: '+name)
            require(frame[base.columns].equals(base),'Feature view differs from canonical manifest: '+name)
            require(set(frame.category)==set(CATEGORIES),'Manifest category mismatch: '+name)
            if 'query_id' in frame:
                require(frame.query_id.is_unique,'Query IDs are not unique.')
                for column,kind,id_column in [('text_feature_index','text','query_id'),('reference_feature_index','reference','reference_image_id'),('target_feature_index','target','target_image_id')]:
                    require(column in frame and frame[column].tolist()==[self.stores[kind].id2idx[v] for v in frame[id_column]],'View mapping mismatch: '+name+'/'+column)
            else:
                require(frame.target_feature_index.tolist()==[self.stores['target'].id2idx[v] for v in frame.image_id],'Gallery image mapping mismatch.')
                for c in CATEGORIES:
                    group=frame[frame.category==c];require(group.image_id.is_unique and group.gallery_index.tolist()==list(range(len(group))),'Gallery order/duplicates mismatch: '+name+'/'+c)
            self.frames[name]=frame;self.manifest_hashes[name]={'canonical':file_hash(base_path),'feature_view':file_hash(path)}
        t=self.frames['official_train_queries'];v=self.frames['official_val_queries'];dt=self.frames['development_train'];dv=self.frames['development_val']
        all_queries=pd.concat([t,v],ignore_index=True)
        require(all_queries.query_id.tolist()==self.stores['text'].ids,'Text query coverage/order mismatch.')
        require(sorted(set(all_queries.reference_image_id))==self.stores['reference'].ids,'Reference feature coverage mismatch.')
        require(sorted(set(t.target_image_id)|set(self.frames['official_val_gallery'].image_id))==self.stores['target'].ids,'Target/gallery feature coverage mismatch.')
        for name,frame in [('official_train',t),('official_val',v)]:
            require(fingerprint(frame.query_id.tolist())==self.protocol['id_order_sha256'][name],'Protocol query order mismatch: '+name)
            require(frame.groupby('category').size().to_dict()==self.protocol['actual_counts'][name],'Protocol query counts mismatch: '+name)
        for c in CATEGORIES:
            gallery=self.frames['official_val_gallery'].loc[lambda x:x.category==c]
            require(fingerprint(gallery.image_id.tolist())==self.protocol['id_order_sha256']['official_val_gallery'][c],'Protocol gallery order mismatch: '+c)
            require(len(gallery)==self.protocol['actual_counts']['official_val_gallery'][c],'Protocol gallery count mismatch: '+c)
        require(set(t.official_split)=={'train'} and set(v.official_split)=={'val'},'Official split mismatch.')
        require(set(dt.query_id).isdisjoint(set(dv.query_id)) and set(dt.query_id)|set(dv.query_id)==set(t.query_id),'Development query coverage overlap/missing.')
        for name,frame in [('development_train',dt),('development_val',dv)]:
            require(frame[t.columns].set_index('query_id').sort_index().equals(t.set_index('query_id').loc[frame.query_id].sort_index()),'Development rows do not match official train: '+name)
        require(not set(dt.target_image_id)&set(dv.target_image_id),'Development target-disjoint requirement failed.')
        require(len(dt)==self.plan['development']['train_count'] and len(dv)==self.plan['development']['val_count'] and len(t)==self.plan['final_training']['actual_query_count'],'Plan/manifest counts mismatch.')
        for qname,gname in [('development_val','development_gallery'),('official_val_queries','official_val_gallery')]:
            q=self.frames[qname];g=self.frames[gname]
            for c in CATEGORIES:require(set(q.loc[q.category==c,'target_image_id'])<=set(g.loc[g.category==c,'image_id']),'Target outside locked gallery: '+qname+'/'+c)
        for c in CATEGORIES:
            actual=self.frames['development_gallery'].loc[lambda x:x.category==c,'image_id'].tolist()
            require(actual==list(dict.fromkeys(dv.loc[dv.category==c,'target_image_id'])),'Development gallery differs from locked unique targets.')
            require(fingerprint(actual)==self.plan['development']['gallery_id_order_sha256'][c],'Development gallery ID-order hash mismatch.')
        self.binding={'feature_fingerprint':self.key,'protocol_sha256':file_hash(self.root/'protocol_config.json'),'training_plan_sha256':file_hash(self.root/'training_plan.json'),
                      'feature_metadata_sha256':file_hash(self.root/'feature_metadata.json'),'architecture_version':fingerprint(self.plan['architecture']),'manifest_hashes':self.manifest_hashes}
        if full_check:self.verify_all()
    def verify_all(self):
        self.full_verified=False
        q=pd.concat([self.frames['official_train_queries'],self.frames['official_val_queries']],ignore_index=True)
        qcols=parse_frame(self.root/'official_train_queries.csv').columns;q=q[list(qcols)]
        gallery=parse_frame(self.root/'official_val_gallery.csv');images=read_json(self.root/'image_content_manifest.json')
        require(fingerprint(images)==self.data_audit['image_content_order_sha256'],'Image content manifest differs from preparation audit.')
        computed=fingerprint({'contract':self.meta['contract'],'query_rows':q.to_dict('records'),'gallery_rows':gallery.to_dict('records'),'images':images})
        require(computed==self.key,'Recomputed preparation fingerprint mismatch; no encoder regeneration permitted here.')
        triplets=defaultdict(list)
        for row in q.itertuples():
            require(row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]),'Canonical triplet key mismatch.')
            triplets[row.triplet_key_sha256].append(row.query_id)
        require(read_json(safe_child(self.feature_root,self.meta['triplet_mapping']))==dict(triplets),'Triplet-to-query mapping mismatch.')
        self.integrity_report={kind:store.audit_all() for kind,store in self.stores.items()};self.full_verified=True;return self.integrity_report

class TripletDataset(Dataset):
    def __init__(self,frame,stores):self.frame=frame;self.stores=stores
    def __len__(self):return len(self.frame)
    def __getitems__(self,indices):
        rows=self.frame.iloc[indices];s=self.stores
        th=s['text'].take_indices(rows.text_feature_index.to_numpy());mask=s['mask'].take_indices(rows.text_feature_index.to_numpy())
        rh=s['reference'].take_indices(rows.reference_feature_index.to_numpy());target=s['target'].take_indices(rows.target_feature_index.to_numpy())
        return [{'text_hidden':torch.from_numpy(th[i]),'text_mask':torch.from_numpy(mask[i]),'reference_hidden':torch.from_numpy(rh[i]),'target':torch.from_numpy(target[i]),
                 'target_index':int(row.target_feature_index),'query_id':row.query_id,'category':row.category} for i,row in enumerate(rows.itertuples())]
    def __getitem__(self,index):return self.__getitems__([index])[0]

def make_loader(frame,package,batch_size,shuffle=False,generator=None,drop_last=False):
    require(batch_size>0 and (not drop_last or len(frame)>=batch_size),'Empty training batches after drop_last.')
    return DataLoader(TripletDataset(frame,package.stores),batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=0,pin_memory=DEVICE.type=='cuda',generator=generator)


In [ ]:
def recall_metrics(ranks):
    ranks=np.asarray(ranks,dtype=np.int64);require(ranks.ndim==1 and len(ranks)>0 and (ranks>=1).all(),'Invalid/empty target ranks.')
    return {f'R@{k}':float((ranks<=k).mean()*100) for k in KS}
def aggregate_categories(metrics):
    require(set(metrics)==set(CATEGORIES),'Macro requires all three categories.')
    overall={f'R@{k}':float(np.mean([metrics[c][f'R@{k}'] for c in CATEGORIES])) for k in KS}
    return {'overall':overall,'Mean':float((overall['R@10']+overall['R@50'])/2)}
def exact_rank(queries,gallery,targets,top_k=50):
    q=normalized(queries);g=normalized(gallery);targets=np.asarray(targets,dtype=np.int64)
    require(len(targets)==len(q) and (targets>=0).all() and (targets<len(g)).all(),'Target gallery position mismatch.')
    scores=q@g.T;target_score=scores[np.arange(len(q)),targets]
    tied=scores==target_score[:,None]
    ranks=1+(scores>target_score[:,None]).sum(1)+(tied&(np.arange(len(g))[None,:]<targets[:,None])).sum(1)
    order=np.argsort(-scores,axis=1,kind='stable')[:,:min(top_k,len(g))]
    return ranks,order,np.take_along_axis(scores,order,axis=1),tied.sum(1)

def evaluate_model(model,package,query_role,gallery_role,export_dir=None):
    require({'development_val':'development_gallery','official_val_queries':'official_val_gallery'}.get(query_role)==gallery_role,'Query/gallery protocol roles mixed.')
    queries=package.frames[query_role];gallery=package.frames[gallery_role];per_category={};category_rows=[];records=[];model.eval()
    for category in CATEGORIES:
        q=queries[queries.category==category];g=gallery[gallery.category==category].sort_values('gallery_index',kind='stable')
        require(len(q)>0 and len(g)>0 and g.gallery_index.tolist()==list(range(len(g))),'Empty category or changed gallery order.')
        ids=g.image_id.tolist();id2pos={v:i for i,v in enumerate(ids)};require(len(ids)==len(id2pos),'Duplicate gallery image.')
        vectors=normalized(package.stores['target'].take_indices(g.target_feature_index.to_numpy()));ranks_for_category=[]
        loader=make_loader(q,package,CFG['eval_batch_size'])
        offset=0
        for batch in loader:
            n=len(batch['query_id']);rows=q.iloc[offset:offset+n];offset+=n
            with torch.inference_mode():
                value=query_forward(model,batch['reference_hidden'].to(DEVICE),batch['text_hidden'].to(DEVICE),batch['text_mask'].to(DEVICE)).float().cpu().numpy()
            require(value.shape==(n,512) and np.isfinite(value).all(),'Query output shape/finite mismatch.')
            targets=[id2pos[x] for x in rows.target_image_id];ranks,top,scores,ties=exact_rank(value,vectors,targets,50)
            ranks_for_category.extend(int(x) for x in ranks)
            if export_dir is not None:
                for row,rank,indices,values,tie_count in zip(rows.itertuples(),ranks,top,scores,ties):
                    records.append({'query_id':row.query_id,'category':category,'reference_image_id':row.reference_image_id,'target_image_id':row.target_image_id,
                       'target_rank':int(rank),'target_score_tie_count':int(tie_count),**{f'hit@{k}':bool(rank<=k) for k in KS},
                       'topk':[{'image_id':ids[int(i)],'gallery_index':int(i),'score':float(score)} for i,score in zip(indices,values)]})
        metrics=recall_metrics(ranks_for_category);per_category[category]=metrics;category_rows.append({'category':category,'n_query':len(q),'n_gallery':len(g),**metrics})
    result={'protocol':'development-internal-per-category-v1' if query_role=='development_val' else package.protocol['protocol'],'unit':'percent','query_role':query_role,'gallery_role':gallery_role,
      'per_category':per_category,**aggregate_categories(per_category),'n_query':len(queries),'query_counts':queries.groupby('category').size().to_dict(),'gallery_counts':gallery.groupby('category').size().to_dict(),
      'search':'exact cosine numpy FP32 matmul, reference retained','tie_break':'score desc, original gallery_index asc for exact FP32 equality'}
    if export_dir is not None:
        export_dir=Path(export_dir);by_id={row['query_id']:row for row in records};require(len(by_id)==len(queries),'Duplicate/lost prediction queries.')
        ordered=[by_id[q] for q in queries.query_id];write_csv(export_dir/'per_query_results.csv',pd.DataFrame([{k:v for k,v in row.items() if k!='topk'} for row in ordered]))
        atomic_bytes(export_dir/'topk_predictions.jsonl',b''.join(canonical(row)+b'\n' for row in ordered))
        write_csv(export_dir/'per_category_metrics.csv',pd.DataFrame(category_rows));write_json(export_dir/'metrics.json',result)
    return result

def validate_metric_object(metric):
    require(metric['unit']=='percent' and set(metric['per_category'])==set(CATEGORIES),'Metric schema/unit mismatch.')
    for c in CATEGORIES:
        values=[metric['per_category'][c][f'R@{k}'] for k in KS];require(all(isinstance(v,(int,float)) and math.isfinite(v) and 0<=v<=100 for v in values),'Metric is missing/nonfinite/outside percent range.')
        require(values==sorted(values),'Recall is not monotonic.')
    recomputed=aggregate_categories(metric['per_category'])
    require(all(abs(metric['overall'][f'R@{k}']-recomputed['overall'][f'R@{k}'])<=1e-10 for k in KS) and abs(metric['Mean']-recomputed['Mean'])<=1e-10,'Macro/Mean not reproducible.')

def validate_final_predictions(folder,package):
    folder=Path(folder);metric=read_json(folder/'metrics.json');validate_metric_object(metric)
    q=package.frames['official_val_queries'];g=package.frames['official_val_gallery'];frame=read_csv(folder/'per_query_results.csv')
    require(frame.query_id.tolist()==q.query_id.tolist() and frame.query_id.is_unique,'Per-query coverage/order mismatch.')
    for col in ('category','reference_image_id','target_image_id'):require(frame[col].tolist()==q[col].tolist(),'Prediction source IDs mismatch: '+col)
    per_category={}
    for c in CATEGORIES:
        rows=frame[frame.category==c];ranks=rows.target_rank.astype(int).to_numpy();require((ranks>=1).all() and (ranks<=int((g.category==c).sum())).all(),'Rank outside category gallery.')
        per_category[c]=recall_metrics(ranks)
        for k in KS:require(rows[f'hit@{k}'].map({'True':True,'False':False}).tolist()==(ranks<=k).tolist(),'Per-query hit flag mismatch.')
    expected={**aggregate_categories(per_category),'per_category':per_category}
    require(expected['per_category']==metric['per_category'] and abs(expected['Mean']-metric['Mean'])<1e-10 and metric['n_query']==len(q),'Metrics disagree with per-query ranks.')
    category_csv=read_csv(folder/'per_category_metrics.csv');require(category_csv.category.tolist()==list(CATEGORIES),'Category CSV order/schema mismatch.')
    for c in CATEGORIES:
        for k in KS:require(abs(float(category_csv.loc[category_csv.category==c,f'R@{k}'].iloc[0])-per_category[c][f'R@{k}'])<1e-9,'Category CSV recall mismatch.')
    for row in category_csv.itertuples():
        c=row.category;require(int(row.n_query)==int((q.category==c).sum()) and int(row.n_gallery)==int((g.category==c).sum()),'Category counts mismatch.')
    predictions=[json.loads(line) for line in (folder/'topk_predictions.jsonl').read_text().splitlines()]
    require([row['query_id'] for row in predictions]==q.query_id.tolist(),'Top-K query coverage/order mismatch.')
    lookups={c:g[g.category==c].sort_values('gallery_index').image_id.tolist() for c in CATEGORIES}
    for source,row in zip(frame.itertuples(),predictions):
        entries=row['topk'];ids=lookups[row['category']];require(len(entries)==min(50,len(ids)) and len({x['image_id'] for x in entries})==len(entries),'Top-K length/duplicate error.')
        require(all(0<=x['gallery_index']<len(ids) and ids[x['gallery_index']]==x['image_id'] and math.isfinite(x['score']) for x in entries),'Top-K ID/index/finite mismatch.')
        require(entries==sorted(entries,key=lambda x:(-x['score'],x['gallery_index'])),'Top-K score/tie order mismatch.')
        rank=int(source.target_rank)
        require((row['target_image_id'] in [x['image_id'] for x in entries])==(rank<=len(entries)),'Target rank/Top-K disagreement.')
        if rank<=len(entries):require(entries[rank-1]['image_id']==row['target_image_id'],'Target position disagrees with rank.')
    return metric


In [ ]:
SEMANTIC_CONTRACT_VERSION='visfit-taskb-head-training-semantics-v1'
LEGACY_IMPLEMENTATION=('visfit-taskb-heads-v1.0.0','32af3b68e6a1e1ce23359f8c5ac1e4c8fbd52a7a9271dc05b14a3932f8f475db')
PRECISION_KEYS=('dtype','amp','matmul_precision','tf32','num_workers')

def semantic_config(config):
    # Library/hardware details stay in provenance. Reading completed baseline is distinct from strict resume.
    ignored={'logic_version','shared_logic_sha256','semantic_contract_version','semantic_contract_sha256','runtime_contract','device_type'}
    result={k:v for k,v in config.items() if k not in ignored}
    result['precision_contract']={k:config['runtime_contract'][k] for k in PRECISION_KEYS}
    result['loss_source']=LOSS_SOURCE_RECORD;result['semantic_contract_version']=SEMANTIC_CONTRACT_VERSION
    return result

def semantic_base(package):
    return {'version':SEMANTIC_CONTRACT_VERSION,'preparation':package.binding,'architecture':package.plan['architecture'],'loss':package.plan['loss'],
            'hyperparameters':package.h,'models':package.plan['models'],'seeds':package.plan['seeds'],'development':package.plan['development'],
            'epoch_lock_rule':package.plan['final_training']['epoch_lock_rule'],'precision':{k:runtime_contract()[k] for k in PRECISION_KEYS},
            'evaluator_source_sha256':EVALUATOR_SOURCE_SHA256,'model_loss_source_sha256':MODEL_LOSS_SOURCE_SHA256,
            'scheduler_rule':'source LambdaLR; optimizer.step, logit_scale clamp, scheduler.step; exact E* rescaling for final',
            'loader_policy':'seeded torch.Generator, shuffle=True, drop_last=True, num_workers=0; source ID order unchanged'}

def to_device(tensor):return tensor.to(DEVICE,non_blocking=DEVICE.type=='cuda' and tensor.is_pinned())
def tensors_are_finite(tensors):
    tensors=list(tensors)
    return bool(tensors) and bool(torch.stack([torch.isfinite(t).all() for t in tensors]).all().item())

def head_update(model,batch,optimizer,scheduler,h):
    optimizer.zero_grad(set_to_none=True)
    composed=query_forward(model,to_device(batch['reference_hidden']),to_device(batch['text_hidden']),to_device(batch['text_mask']))
    target=to_device(batch['target']);ids=to_device(batch['target_index'])
    loss=multi_positive_infonce(composed,target,ids,model.logit_scale)
    # One loss transfer; gradient/weight predicates each transfer ONE combined scalar, retaining all checks.
    loss_value=float(loss.detach().item());require(math.isfinite(loss_value),'Nonfinite training loss.')
    loss.backward();grads=[p.grad for p in model.parameters() if p.grad is not None]
    require(tensors_are_finite(grads),'Nonfinite/missing gradients.')
    torch.nn.utils.clip_grad_norm_(model.parameters(),h['gradient_clip_norm'],error_if_nonfinite=True)
    optimizer.step()
    with torch.no_grad():model.logit_scale.clamp_(*h['logit_scale_clamp'])
    scheduler.step()
    require(tensors_are_finite(model.parameters()),'Nonfinite model weights after update.')
    return loss_value

class EpochTiming:
    def __init__(self):
        self.start=time.perf_counter();self.data_s=0.;self.compute_host_s=0.;self.events=[];self.compute_gpu_s=None
    def batches(self,loader):
        clock=time.perf_counter();iterator=iter(loader);self.data_s+=time.perf_counter()-clock
        while True:
            clock=time.perf_counter()
            try:batch=next(iterator)
            except StopIteration:return
            self.data_s+=time.perf_counter()-clock;yield batch
    @contextmanager
    def compute(self):
        clock=time.perf_counter()
        if DEVICE.type=='cuda':
            begin=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True);begin.record()
        try:yield
        finally:
            if DEVICE.type=='cuda':end.record();self.events.append((begin,end))
            self.compute_host_s+=time.perf_counter()-clock
    def collect_compute(self):
        if self.events:
            torch.cuda.synchronize(DEVICE) # Once at the train/validation boundary; never per batch for timing.
            self.compute_gpu_s=sum(a.elapsed_time(b) for a,b in self.events)/1000
    def record(self,epoch,validation_s,save_s):
        return {'epoch':epoch,'data_loading_cpu_s':self.data_s,'forward_backward_update_host_s':self.compute_host_s,
          'forward_backward_update_cuda_stream_s':self.compute_gpu_s,'validation_wall_s':validation_s,'checkpoint_and_artifact_save_s':save_s,
          'epoch_wall_s':time.perf_counter()-self.start,'timing_method':'perf_counter host; per-batch CUDA events collected once at epoch phase boundary; no timing-only per-batch synchronization. CUDA stream time includes transfers and stream idle gaps; not kernel-only.',
          'timing_artifact_write_included':False,'status':'measured'}

def load_timing_rows(folder,completed_epoch):
    path=Path(folder)/'timing_history.json';rows=read_json(path) if path.is_file() else []
    require(len({r['epoch'] for r in rows})==len(rows),'Duplicate timing epochs.')
    # A crash may commit checkpoint before the independent timing sidecar. Do not invent measurements.
    known={r['epoch']:r for r in rows if 1<=r['epoch']<=completed_epoch}
    return [known.get(e,{'epoch':e,'status':'unavailable_interrupted_timing_write'}) for e in range(1,completed_epoch+1)]
def save_timing_rows(folder,rows):
    write_json(Path(folder)/'timing_history.json',rows);write_csv(Path(folder)/'timing_history.csv',pd.DataFrame(rows))

def short_profile(package):
    n=CFG['profile_batches'];require(type(n) is int and 0<=n<=10,'profile_batches must be 0..10.')
    if n==0:return {'status':'skipped','reason':'profile_batches=0','main_training_changed':False}
    before=state_digest(capture_rng());timer=EpochTiming();losses=[];seen=[]
    with preserve_rng():
        seed=package.plan['seeds'][0];set_seed(seed);model=build_model(package.plan,'xaa_full_dvr').float().to(DEVICE).train()
        generator=torch.Generator().manual_seed(seed);loader=make_loader(package.frames['development_train'],package,package.h['train_batch_size'],True,generator,True)
        optimizer,scheduler,_=optimizer_scheduler(model,package.h,package.h['max_development_epochs'],len(loader))
        for _,batch in zip(range(n),timer.batches(loader)):
            with timer.compute():losses.append(head_update(model,batch,optimizer,scheduler,package.h))
            seen.extend(batch['query_id'])
        timer.collect_compute()
        report={'status':'passed','temporary_model':'xaa_full_dvr','batches':len(losses),'samples':len(seen),'loss_values_diagnostic_only':losses,'query_order_sha256':fingerprint(seen),
                'timing':timer.record(0,0.,0.),'main_training_changed':False,'not_a_Colab_speedup_measurement':True}
        del optimizer,scheduler,model
    require(state_digest(capture_rng())==before,'Profiler changed main RNG state.');report['rng_restored']=True;return report


In [ ]:
def baseline_campaign():return Path(CFG['baseline_campaign']).resolve()

def check_baseline_identity(package):
    root=baseline_campaign();identity=read_json(root/'campaign_identity.json')
    require((identity['logic_version'],identity['shared_logic_sha256'])==LEGACY_IMPLEMENTATION,'Unrecognized baseline campaign implementation; only the verified original v1.0.0 hash may be read.')
    require(identity['preparation']==package.binding and identity['hyperparameters']==package.h and identity['models']==package.plan['models'] and identity['seeds']==package.plan['seeds'],'Baseline campaign feature/protocol/manifest/hyperparameter/seed mismatch.')
    require(all(identity['runtime_contract'][k]==runtime_contract()[k] for k in PRECISION_KEYS),'Baseline precision/worker contract differs.')
    return identity

def verify_known_baseline(folder,package,seed):
    # Read-only completed-result validation. NEVER authorize loading these states into a training/resume run.
    check_baseline_identity(package);folder=Path(folder);config=read_json(folder/'training_config.json')
    require((config['logic_version'],config['shared_logic_sha256'])==LEGACY_IMPLEMENTATION,'Baseline config implementation is not allowlisted.')
    expected=training_config(package,'xaa_noft',seed,'development',package.h['max_development_epochs'])
    require(semantic_config(config)==semantic_config(expected),'Baseline semantic training contract differs from the locked plan/features/protocol/architecture/loss.')
    signature=fingerprint(config);result=verify_completion(folder,signature,package,'development',purpose='baseline_result')
    validate_development_evidence(folder,package,config,result,purpose='baseline_result')
    return result,{'usage':'completed_baseline_read_only_no_resume','logic_version':config['logic_version'],'shared_logic_sha256':config['shared_logic_sha256'],
                   'semantic_contract_sha256':fingerprint(semantic_config(config)),'recorded_runtime':config['runtime_contract'],'recorded_device_type':config['device_type']}

def validate_development_evidence(folder,package,config,result,purpose='resume'):
    signature=fingerprint(config);last=checked_checkpoint(Path(folder)/'last_checkpoint.pt',signature,purpose)
    require(last['config']==config and last['seed']==config['seed'] and last['phase']=='development','Development checkpoint config/seed/phase mismatch.')
    require(last['global_step']==last['epoch']*(len(package.frames['development_train'])//package.h['train_batch_size']),'Development epoch/global-step mismatch.')
    history=last['history'];require([r['epoch'] for r in history]==list(range(1,last['epoch']+1)),'Incomplete/unordered epoch history.')
    expected_rows=[{'epoch':r['epoch'],'metrics':r['development_metrics']} for r in history if r['development_metrics'] is not None]
    require(result['per_epoch']==expected_rows,'Metrics differ from checkpoint history; never fill missing evaluations.')
    require(result['model']==config['model'] and result['seed']==config['seed'] and result['epochs_trained']==last['epoch'],'Development result model/seed/epoch mismatch.')
    require(result['binding']==package.binding and result['hyperparameters']==package.h and not result['official_val_used'],'Development result binding/policy mismatch.')
    cadence=package.h['eval_every_epochs']
    require([r['epoch'] for r in expected_rows]==list(range(cadence,last['epoch']+1,cadence)),'Real evaluation cadence differs from plan.')
    for r in expected_rows:
        metric=r['metrics'];validate_metric_object(metric)
        require(metric['query_role']=='development_val' and metric['gallery_role']=='development_gallery','Development metric used the wrong source.')
        require(metric['n_query']==len(package.frames['development_val']) and metric['query_counts']==package.frames['development_val'].groupby('category').size().to_dict() and metric['gallery_counts']==package.frames['development_gallery'].groupby('category').size().to_dict(),'Development metric coverage differs from locked manifests.')
    best=max(r['metrics']['Mean'] for r in expected_rows);earliest=min(r['epoch'] for r in expected_rows if r['metrics']['Mean']==best)
    require(last['best_epoch']==result['best_epoch']==earliest and last['best_development_metric']==result['best_Mean']==best,'Best epoch/metric/earliest tie differs.')
    best_state=checked_checkpoint(Path(folder)/'best_checkpoint.pt',signature,purpose);require(best_state['model_state'].keys()==last['model_state'].keys(),'Best/last model keys differ.')
    require(best_state['payload_digest']==last['best_snapshot']['payload_digest'] and state_digest(best_state)==state_digest(last['best_snapshot']),'Best checkpoint differs from authoritative snapshot.')
    for state in (last,best_state):
        require(tensors_are_finite(state['model_state'].values()),'Nonfinite saved head weights.')
        require(all(k in state for k in ('optimizer_state','scheduler_state','rng_states','initialization_hash')),'Missing required checkpoint state.')
        require(state['scheduler_state']['last_epoch']==state['global_step'],'Scheduler/global step mismatch.')
    frame=read_csv(Path(folder)/'training_history.csv');require(frame.epoch.astype(int).tolist()==[r['epoch'] for r in history],'CSV/checkpoint epoch coverage mismatch.')
    for source,row in zip(history,frame.to_dict('records')):
        require(math.isfinite(float(source['train_loss'])) and abs(float(row['train_loss'])-source['train_loss'])<1e-10,'CSV/checkpoint training loss mismatch.')
        metric=source['development_Mean'];require(row['development_Mean']=='' if metric is None else abs(float(row['development_Mean'])-metric)<1e-10,'CSV/checkpoint development metric mismatch.')
    provenance=read_json(Path(folder)/'provenance.json')
    require(provenance['initialization_hash']==last['initialization_hash'] and provenance['architecture_sources']==ARCHITECTURE_SOURCES and provenance['loss_source']==LOSS_SOURCE_RECORD and provenance['warm_start'] is False and provenance['encoder_training'] is False,'Provenance source/init/warm-start mismatch.')
    policy=read_json(Path(folder)/'scheduler_policy.json')
    with preserve_rng():
        model=build_model(package.plan,config['model']);_,_,expected_policy=optimizer_scheduler(model,package.h,config['epoch_budget'],len(package.frames['development_train'])//package.h['train_batch_size'])
    require(policy==expected_policy,'Scheduler policy differs from original source rule.')
    return True

def inventory_legacy_training(package):
    root=baseline_campaign();records=[]
    # These old DVR/final runs are never silently imported or resumed. New optimized output is isolated.
    for phase in ('development','final'):
        for name in package.plan['models']:
            if phase=='development' and name=='xaa_noft':continue
            for seed in package.plan['seeds']:
                folder=run_directory(root,phase,name,seed)
                if (folder/'last_checkpoint.pt').is_file() or (folder/'completion.json').is_file():
                    cfg=read_json(folder/'training_config.json');known=(cfg.get('logic_version'),cfg.get('shared_logic_sha256'))==LEGACY_IMPLEMENTATION
                    records.append({'folder':str(folder),'implementation_recognized':known,'resume_in_optimized_notebook':'blocked_no_automatic_migration','action':'Resume using the original matching notebook; optimized training uses the new output campaign. Old checkpoint/result remains unchanged.'})
    return records


In [ ]:
def state_digest(value):
    h=hashlib.sha256()
    def visit(x):
        if isinstance(x,torch.Tensor):
            t=x.detach().cpu().contiguous();h.update(str(t.dtype).encode());h.update(canonical(list(t.shape)));h.update(t.numpy().tobytes())
        elif isinstance(x,dict):
            for key in sorted(x,key=lambda v:(type(v).__name__,str(v))):
                if key=='payload_digest':continue
                h.update(canonical([type(key).__name__,key]));visit(x[key])
        elif isinstance(x,(tuple,list)):
            h.update(type(x).__name__.encode());h.update(str(len(x)).encode())
            for v in x:visit(v)
        else:h.update(canonical(x))
    visit(value);return h.hexdigest()
def checked_checkpoint(path,signature=None,purpose='resume'):
    state=safe_checkpoint(path);implementation=(state.get('logic_version'),state.get('shared_logic_sha256'))
    current=implementation==(LOGIC_VERSION,SHARED_LOGIC_SHA256)
    legacy_baseline=purpose=='baseline_result' and implementation==LEGACY_IMPLEMENTATION and state.get('phase')=='development' and state.get('config',{}).get('model')=='xaa_noft'
    require(current or legacy_baseline,'Checkpoint implementation mismatch. Completed recognized baseline may be read for aggregation; legacy DVR/final checkpoint cannot be silently resumed. Use original notebook to resume that run, or a fresh optimized campaign.')
    require(state.get('payload_digest')==state_digest(state),'Checkpoint payload checksum mismatch: '+str(path))
    require(fingerprint(state['config'])==state['signature'],'Checkpoint config does not reproduce its signature.')
    if signature is not None:require(state['signature']==signature,'Resume config/seed/manifest/feature signature mismatch.')
    return state

def optimizer_scheduler(model,h,epochs,steps_per_epoch):
    require(epochs>0 and steps_per_epoch>0,'Empty training budget.')
    optimizer=torch.optim.AdamW(model.parameters(),lr=h['learning_rate'],weight_decay=h['weight_decay'],betas=tuple(h['betas']),eps=h['eps'])
    total_steps=epochs*steps_per_epoch;warmup=max(1,int(h['warmup_ratio']*total_steps))
    def lr_lambda(step):
        if step<warmup:return (step+1)/warmup
        progress=(step-warmup)/max(1,total_steps-warmup)
        return 0.5*(1+math.cos(math.pi*progress))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
    return optimizer,scheduler,{'kind':h['scheduler'],'total_steps':total_steps,'warmup_steps':warmup,'steps_per_epoch':steps_per_epoch,
      'formula':'(step+1)/warmup for step<warmup, else 0.5*(1+cos(pi*(step-warmup)/(total-warmup)))','step_order':'optimizer.step then scheduler.step after each batch'}

def campaign_path():
    root=Path(CFG['output_root']).resolve();campaign=(root/CFG['campaign_id']).resolve();require(campaign.parent==root,'campaign_id must be a single directory name.');return campaign

def initialize_campaign(package):
    require(package.full_verified,'Verify every preparation shard before starting training.')
    campaign=campaign_path();prep=package.root
    require(campaign!=baseline_campaign() and baseline_campaign() not in campaign.parents and campaign not in baseline_campaign().parents,'Optimized output must be separate from the read-only baseline campaign.')
    for source in (prep,Path('/content/drive/MyDrive/ĐATN/data/processed/models'),Path('/content/drive/MyDrive/ĐATN/data/processed/embeddings')):
        p=source.resolve();require(campaign!=p and p not in campaign.parents and campaign not in p.parents,'Output overlaps preparation/legacy source: '+str(p))
    identity={'logic_version':LOGIC_VERSION,'shared_logic_sha256':SHARED_LOGIC_SHA256,'preparation':package.binding,'runtime_contract':runtime_contract(),
      'models':package.plan['models'],'seeds':package.plan['seeds'],'hyperparameters':package.h,'semantic_contract':semantic_base(package),'baseline_source_campaign':str(Path(CFG['baseline_campaign']).resolve()),'legacy_resume_policy':'No automatic migration; optimized checkpoints resume only with exact implementation/config/runtime. Legacy DVR/final stays untouched.', 'scheduler_final_rule':'Recompute total_steps=E*floor(all_official_train_rows/batch_size), same source warmup+cosine formula.'}
    campaign.mkdir(parents=True,exist_ok=True);write_json(campaign/'campaign_identity.json',identity,immutable=True);return campaign

def run_directory(campaign,phase,name,seed):return Path(campaign)/phase/name/f'seed_{seed}'

def training_config(package,name,seed,phase,epoch_budget,lock=None):
    frame_role='development_train' if phase=='development' else 'official_train_queries';n=len(package.frames[frame_role]);h=package.h
    config={'logic_version':LOGIC_VERSION,'shared_logic_sha256':SHARED_LOGIC_SHA256,'phase':phase,'model':name,'model_spec':package.plan['models'][name],'seed':seed,
      'preparation':package.binding,'architecture':package.plan['architecture'],'hyperparameters':h,'epoch_budget':epoch_budget,'training_role':frame_role,
      'dataset_rows':n,'samples_per_epoch':n//h['train_batch_size']*h['train_batch_size'],'drop_last_train':h['drop_last_train'],
      'runtime_contract':runtime_contract(),'device_type':DEVICE.type,'num_workers':0,'amp_enabled':False,
      'checkpoint_selection':'development per-category Macro Mean, strict improvement / earliest tie' if phase=='development' else 'exact locked epoch E*, no official-val selection',
      'final_lock_sha256':fingerprint(lock) if lock is not None else None,'initialization':'fresh head for each seed/config; no baseline/development checkpoint warm-start',
      'no_encoder_loaded_or_trained':True,'query_forward_inputs':['reference_hidden','text_hidden','text_mask']}
    config['semantic_contract_version']=SEMANTIC_CONTRACT_VERSION;config['semantic_contract_sha256']=fingerprint(semantic_config(config));return config

def write_derived_epoch_artifacts(folder,last):
    rows=[]
    for source in last['history']:
        row={k:v for k,v in source.items() if k!='development_metrics'};metric=source['development_metrics']
        for c in CATEGORIES:
            for k in KS:row[f'{c}_R@{k}']=None if metric is None else metric['per_category'][c][f'R@{k}']
        rows.append(row)
    write_csv(folder/'training_history.csv',pd.DataFrame(rows))
    best=last.get('best_snapshot')
    if best is not None:
        path=folder/'best_checkpoint.pt';marker=folder/'best_checkpoint_integrity.json';valid=False
        if path.is_file() and marker.is_file():
            m=read_json(marker);valid=m.get('epoch')==best['epoch'] and m.get('signature')==best['signature'] and file_hash(path)==m.get('sha256')
        if not valid:
            atomic_torch_save(path,best);write_json(marker,{'epoch':best['epoch'],'signature':best['signature'],'sha256':file_hash(path)})

def checkpoint_outputs(folder,files):
    return {str(name):file_hash(folder/name) for name in files}
def verify_completion(folder,expected_signature,package,phase,purpose='resume'):
    folder=Path(folder);done=read_json(folder/'completion.json');require(done['signature']==expected_signature and done['phase']==phase,'Completion signature/phase mismatch.')
    for name,sha in done['outputs'].items():require(file_hash(safe_child(folder,name))==sha,'Completed artifact checksum mismatch: '+name)
    last=checked_checkpoint(folder/'last_checkpoint.pt',expected_signature,purpose);require(last['finished'] and last['phase']==phase,'Incomplete last checkpoint behind completion marker.')
    require(last['config']['preparation']==package.binding,'Completed run preparation changed.')
    with preserve_rng():
        model=build_model(package.plan,last['config']['model']);model.load_state_dict(last['model_state'],strict=True)
    if phase=='development':
        metrics=read_json(folder/'development_metrics.json');require(metrics['source']=='development_val' and metrics['signature']==expected_signature,'Development result source/signature mismatch.')
        require([row['epoch'] for row in metrics['per_epoch']]==[row['epoch'] for row in last['history'] if row['development_metrics'] is not None],'Development epoch coverage mismatch.')
        for row in metrics['per_epoch']:validate_metric_object(row['metrics'])
        best=checked_checkpoint(folder/'best_checkpoint.pt',expected_signature,purpose);require(best['epoch']==last['best_epoch'] and best['best_development_metric']==last['best_development_metric'],'Best checkpoint mismatch.')
        return metrics
    final=checked_checkpoint(folder/'final_checkpoint.pt',expected_signature,purpose);require(final['epoch']==last['config']['epoch_budget'] and final['global_step']==last['global_step'],'Final checkpoint epoch mismatch.')
    return validate_final_predictions(folder,package)

def train_run(package,campaign,name,seed,phase,lock=None,stop_after_epoch=None):
    require(package.full_verified,'Full package integrity validation required.')
    require(name in package.plan['models'] and seed in package.plan['seeds'],'Model/seed not in locked plan.')
    require(phase in ('development','final'),'Unknown training phase.')
    require(not (phase=='development' and name=='xaa_noft'),'Optimized notebooks never retrain development baseline; read the verified original result only.')
    if phase=='final':require(lock is not None and lock['selected_epoch']>0,'Final lock required.')
    epochs=package.h['max_development_epochs'] if phase=='development' else lock['selected_epoch']
    config=training_config(package,name,seed,phase,epochs,lock);signature=fingerprint(config);folder=run_directory(campaign,phase,name,seed)
    if folder.exists():require((folder/'training_config.json').is_file(),'Existing unrecognized output directory; do not overwrite.')
    folder.mkdir(parents=True,exist_ok=True);write_json(folder/'training_config.json',config,immutable=True)
    if (folder/'completion.json').is_file():return verify_completion(folder,signature,package,phase)
    set_seed(seed);generator=torch.Generator().manual_seed(seed);model=build_model(package.plan,name).float().to(DEVICE)
    initial_hash=model_hash(model);role=config['training_role'];loader=make_loader(package.frames[role],package,package.h['train_batch_size'],True,generator,True)
    optimizer,scheduler,schedule=optimizer_scheduler(model,package.h,epochs,len(loader));write_json(folder/'scheduler_policy.json',schedule,immutable=True)
    history=[];timing_rows=[];global_step=0;best_metric=None;best_epoch=None;best_snapshot=None;no_improve=0;start_epoch=1;finished=False
    path=folder/'last_checkpoint.pt'
    if path.exists():
        require(CFG['resume'],'Last checkpoint exists and resume=False; use a new campaign.')
        last=checked_checkpoint(path,signature);require(last['epoch']<=epochs and last['global_step']==last['epoch']*len(loader),'Not a valid epoch-boundary checkpoint.')
        model.load_state_dict(last['model_state'],strict=True);optimizer.load_state_dict(last['optimizer_state']);scheduler.load_state_dict(last['scheduler_state'])
        history=last['history'];global_step=last['global_step'];best_metric=last['best_development_metric'];best_epoch=last['best_epoch'];best_snapshot=last['best_snapshot'];no_improve=last['no_improve']
        initial_hash=last['initialization_hash'];start_epoch=last['epoch']+1;finished=last['finished'];restore_rng(last['rng_states'],generator);write_derived_epoch_artifacts(folder,last)
    timing_rows=load_timing_rows(folder,start_epoch-1)
    write_json(folder/'provenance.json',{'config_signature':signature,'preparation':package.binding,'initialization_hash':initial_hash,'environment':execution_environment(),'architecture_sources':ARCHITECTURE_SOURCES,'loss_source':LOSS_SOURCE_RECORD,
      'phase':phase,'seed':seed,'model':name,'warm_start':False,'checkpoint_resume':path.exists(),'encoder_training':False,'implementation':{'logic_version':LOGIC_VERSION,'shared_logic_sha256':SHARED_LOGIC_SHA256},'semantic_contract_sha256':config['semantic_contract_sha256']})
    if not finished:
        for epoch in range(start_epoch,epochs+1):
            timer=EpochTiming();model.train();weighted=0.;samples=0;lr_before=float(optimizer.param_groups[0]['lr'])
            for batch in timer.batches(loader):
                with timer.compute():loss_value=head_update(model,batch,optimizer,scheduler,package.h)
                global_step+=1;n=len(batch['query_id']);weighted+=loss_value*n;samples+=n
            timer.collect_compute();validation_clock=time.perf_counter()
            require(samples==config['samples_per_epoch'] and samples>0,'Epoch sample coverage differs from locked drop_last policy.')
            metric=None;improved=False
            if phase=='development' and epoch%package.h['eval_every_epochs']==0:
                metric=evaluate_model(model,package,'development_val','development_gallery');validate_metric_object(metric)
                improved=best_metric is None or metric['Mean']>best_metric
                if improved:best_metric=metric['Mean'];best_epoch=epoch;no_improve=0
                else:no_improve+=1
            if DEVICE.type=='cuda':torch.cuda.synchronize(DEVICE) # At validation phase boundary, not per batch.
            validation_s=time.perf_counter()-validation_clock;save_clock=time.perf_counter()
            early=phase=='development' and metric is not None and no_improve>=package.h['patience_evaluations'];finished=early or epoch==epochs
            history.append({'epoch':epoch,'global_step':global_step,'train_loss':weighted/samples,'samples_seen':samples,'lr_first_batch':lr_before,'lr_next_step':float(optimizer.param_groups[0]['lr']),
              'development_Mean':None if metric is None else metric['Mean'],'is_best':improved,'development_metrics':metric})
            base={'logic_version':LOGIC_VERSION,'shared_logic_sha256':SHARED_LOGIC_SHA256,'signature':signature,'phase':phase,'config':config,'seed':seed,
                  'model_state':cpu_clone(model.state_dict()),'optimizer_state':cpu_clone(optimizer.state_dict()),'scheduler_state':cpu_clone(scheduler.state_dict()),'amp_scaler_state':None,
                  'epoch':epoch,'global_step':global_step,'history':copy.deepcopy(history),'best_development_metric':best_metric,'best_epoch':best_epoch,'no_improve':no_improve,
                  'rng_states':capture_rng(generator),'initialization_hash':initial_hash,'finished':finished,'termination_reason':'early_stopping' if early else 'epoch_budget' if finished else 'in_progress'}
            base['payload_digest']=state_digest(base)
            if improved:best_snapshot=cpu_clone(base)
            last={**base,'best_snapshot':best_snapshot};last['payload_digest']=state_digest(last)
            atomic_torch_save(path,last);write_derived_epoch_artifacts(folder,last)
            timing_rows.append(timer.record(epoch,validation_s,time.perf_counter()-save_clock));save_timing_rows(folder,timing_rows)
            print(name,'seed',seed,'epoch',epoch,'loss',round(weighted/samples,5),'dev Mean',None if metric is None else round(metric['Mean'],4))
            if stop_after_epoch is not None and epoch>=stop_after_epoch and not finished:return {'status':'interrupted_for_fixture','epoch':epoch}
            if finished:break
    last=checked_checkpoint(path,signature);write_derived_epoch_artifacts(folder,last)
    save_timing_rows(folder,load_timing_rows(folder,last['epoch']))
    if phase=='development':
        require(last['best_epoch'] is not None,'No development metric within budget; no completed result/lock.')
        metrics={'signature':signature,'source':'development_val','selection_metric':package.plan['development']['checkpoint_metric'],'seed':seed,'model':name,'phase':'development',
                 'best_epoch':last['best_epoch'],'best_Mean':last['best_development_metric'],'epochs_trained':last['epoch'],'termination_reason':last['termination_reason'],
                 'per_epoch':[{'epoch':r['epoch'],'metrics':r['development_metrics']} for r in last['history'] if r['development_metrics'] is not None],
                 'binding':package.binding,'epoch_budget':epochs,'hyperparameters':package.h,'official_val_used':False}
        write_json(folder/'development_metrics.json',metrics);files=['training_config.json','training_history.csv','scheduler_policy.json','best_checkpoint.pt','last_checkpoint.pt','best_checkpoint_integrity.json','development_metrics.json','provenance.json']
    else:
        require(last['epoch']==epochs,'Final training has not reached locked E*.')
        finalization_clock=time.perf_counter();final_path=folder/'final_checkpoint.pt'
        atomic_bytes(final_path,path.read_bytes())
        write_json(folder/'training_completion.json',{'signature':signature,'epoch':epochs,'outputs':checkpoint_outputs(folder,['last_checkpoint.pt','final_checkpoint.pt','training_history.csv'])})
        model.load_state_dict(last['model_state'],strict=True)
        export_save_s=time.perf_counter()-finalization_clock;official_val_clock=time.perf_counter()
        metrics=evaluate_model(model,package,'official_val_queries','official_val_gallery',folder);validate_final_predictions(folder,package)
        if DEVICE.type=='cuda':torch.cuda.synchronize(DEVICE)
        write_json(folder/'finalization_timing.json',{'checkpoint_export_save_s':export_save_s,'official_val_evaluation_and_artifact_wall_s':time.perf_counter()-official_val_clock,'timing_method':'perf_counter with CUDA synchronized only at final validation boundary'})
        files=['finalization_timing.json','training_config.json','training_history.csv','scheduler_policy.json','last_checkpoint.pt','final_checkpoint.pt','training_completion.json','metrics.json','per_query_results.csv','per_category_metrics.csv','topk_predictions.jsonl','provenance.json']
    files+=['timing_history.json','timing_history.csv']
    write_json(folder/'completion.json',{'signature':signature,'phase':phase,'outputs':checkpoint_outputs(folder,files)},immutable=True)
    return verify_completion(folder,signature,package,phase)


In [ ]:
def inspect_development(campaign,package):
    runs={};rows=[];missing=[];invalid=[]
    for name in package.plan['models']:
        for seed in package.plan['seeds']:
            key=f'{name}/seed_{seed}';source_campaign=baseline_campaign() if name=='xaa_noft' else Path(campaign);folder=run_directory(source_campaign,'development',name,seed)
            row={'model_id':name,'model':package.plan['models'][name]['label'],'seed':seed,'status':'not_started','best_epoch':None,'best_development_Mean':None,'epochs_trained':None,'reason':''}
            if not (folder/'completion.json').is_file():
                if (folder/'last_checkpoint.pt').is_file():row['status']='incomplete'
                missing.append(key)
            else:
                try:
                    if name=='xaa_noft':result,implementation=verify_known_baseline(folder,package,seed)
                    else:
                        config=training_config(package,name,seed,'development',package.h['max_development_epochs']);signature=fingerprint(config)
                        result=verify_completion(folder,signature,package,'development');validate_development_evidence(folder,package,config,result)
                        implementation={'usage':'optimized_training','logic_version':LOGIC_VERSION,'shared_logic_sha256':SHARED_LOGIC_SHA256,'semantic_contract_sha256':config['semantic_contract_sha256']}
                    row['implementation']=implementation['logic_version'];row['source_campaign']=str(source_campaign)
                    require(result['binding']==package.binding and result['hyperparameters']==package.h and not result['official_val_used'],'Development provenance/hyperparameters mismatch.')
                    epochs=[r['epoch'] for r in result['per_epoch']];require(epochs==sorted(set(epochs)) and all(e%package.h['eval_every_epochs']==0 and 1<=e<=result['epochs_trained'] for e in epochs),'Invalid development evaluation epochs.')
                    require(result['epoch_budget']==package.h['max_development_epochs'],'Development budget mismatch.')
                    best=max(r['metrics']['Mean'] for r in result['per_epoch']);best_epoch=min(r['epoch'] for r in result['per_epoch'] if r['metrics']['Mean']==best)
                    require(result['best_Mean']==best and result['best_epoch']==best_epoch,'Development best checkpoint/earliest tie mismatch.')
                    row.update(status='development_complete',best_epoch=best_epoch,best_development_Mean=best,epochs_trained=result['epochs_trained'])
                    files={f:file_hash(folder/f) for f in ('completion.json','development_metrics.json','training_history.csv','training_config.json','best_checkpoint.pt','last_checkpoint.pt')}
                    runs[key]={'model':name,'seed':seed,'result':result,'result_hashes':files,'relative_dir':str(folder.relative_to(source_campaign)),'source_campaign':str(source_campaign.resolve()),'implementation':implementation}
                except Exception as exc:row.update(status='invalid',reason=redact(exc));invalid.append(key)
            rows.append(row)
    return runs,pd.DataFrame(rows),missing,invalid

def select_common_epoch(run_metrics,required_keys):
    """Pure development-only selector; missing runs/epochs are never imputed or replaced by best-at-another-epoch."""
    missing=sorted(set(required_keys)-set(run_metrics))
    if missing:return {'ready':False,'missing_runs':missing,'eligible_epochs':[],'per_epoch':[],'selected_epoch':None}
    by_run={key:{row['epoch']:row['metrics']['Mean'] for row in run_metrics[key]['per_epoch']} for key in required_keys}
    for key,table in by_run.items():
        require(len(table)==len(run_metrics[key]['per_epoch']),'Duplicate evaluated epochs: '+key)
        require(all(type(e) is int and e>0 and isinstance(v,(int,float)) and math.isfinite(v) and 0<=v<=100 for e,v in table.items()),'Invalid/missing development metric: '+key)
    eligible=sorted(set.intersection(*(set(table) for table in by_run.values())))
    records=[{'epoch':e,'run_metrics':{key:by_run[key][e] for key in required_keys},'mean_development_Macro_Mean':float(np.mean([by_run[key][e] for key in required_keys],dtype=np.float64))} for e in eligible]
    if not eligible:return {'ready':False,'missing_runs':[],'eligible_epochs':[],'per_epoch':[],'selected_epoch':None,'reason':'No common epoch with real metrics in every required completed run.'}
    best=max(r['mean_development_Macro_Mean'] for r in records);chosen=min(r['epoch'] for r in records if r['mean_development_Macro_Mean']==best)
    return {'ready':True,'missing_runs':[],'eligible_epochs':eligible,'per_epoch':records,'selected_epoch':chosen,'selected_mean':best,
            'selection_data':'development only','tie_policy':'earliest epoch on exact equality of float64 mean','imputation':False}

def expected_final_lock(package,runs,selection):
    require(selection['ready'],'Cannot create final lock from incomplete development.')
    return {'schema_version':'final-training-lock-v2-ioopt','logic_version':LOGIC_VERSION,'shared_logic_sha256':SHARED_LOGIC_SHA256,
      'semantic_contract':semantic_base(package),'selected_epoch':selection['selected_epoch'],'eligible_epochs':selection['eligible_epochs'],'selection':selection,
      'models':package.plan['models'],'seeds':package.plan['seeds'],'hyperparameters':package.h,
      'preparation':package.binding,'development_results':{key:{'source_campaign':row['source_campaign'],'relative_dir':row['relative_dir'],'hashes':row['result_hashes'],'implementation':row['implementation']} for key,row in sorted(runs.items())},
      'official_val_metrics_used':False,'warm_start':False,
      'final_scheduler':{'formula':'source LambdaLR: linear (step+1)/warmup then cosine','total_steps_rule':'E* * floor(len(official_train_queries) / train_batch_size)',
           'warmup_steps_rule':'max(1, int(warmup_ratio * total_steps))','total_steps':selection['selected_epoch']*(len(package.frames['official_train_queries'])//package.h['train_batch_size']),
           'warmup_ratio':package.h['warmup_ratio'],'same_rule_for_all_models_and_seeds':True},
      'epoch_lock_rule_from_preparation':package.plan['final_training']['epoch_lock_rule'],'checkpoint_selection':'exact E* for every final model/seed; no official-val early stopping or seed selection'}

def refresh_development_summary(campaign,package,create_lock=True):
    folder=Path(campaign)/'development';folder.mkdir(exist_ok=True)
    runs,summary,missing,invalid=inspect_development(campaign,package);write_csv(folder/'development_summary.csv',summary)
    required=[f'{name}/seed_{seed}' for name in package.plan['models'] for seed in package.plan['seeds']]
    selection=select_common_epoch({k:v['result'] for k,v in runs.items()},required)
    selection['invalid_runs']=invalid;selection['missing_or_incomplete_runs']=missing
    write_json(folder/'epoch_selection.json',selection)
    epoch_rows=[{'epoch':r['epoch'],'mean_development_Macro_Mean':r['mean_development_Macro_Mean'],'selected':r['epoch']==selection['selected_epoch']} for r in selection['per_epoch']]
    write_csv(folder/'epoch_selection.csv',pd.DataFrame(epoch_rows,columns=['epoch','mean_development_Macro_Mean','selected']))
    ready=selection['ready'] and not missing and not invalid
    if ready and create_lock:write_json(folder/'final_training_lock.json',expected_final_lock(package,runs,selection),immutable=True)
    write_json(folder/'lock_status.json',{'ready':ready,'lock_written_or_verified':ready and create_lock,'eligible_epochs':selection['eligible_epochs'],'missing':missing,'invalid':invalid,'official_val_used':False})
    note=f"# Development\n- Completed runs: {len(runs)}/{len(required)}.\n- Eligible common epochs: {selection['eligible_epochs']}.\n- E*: {selection['selected_epoch'] if ready else 'pending'}.\n- Selection uses development per-category Macro Mean only; missing metrics are never filled.\n"
    atomic_bytes(folder/'summary.md',note.encode());return summary,selection

def load_final_lock(campaign,package):
    path=Path(campaign)/'development/final_training_lock.json';require(path.is_file(),'Final lock missing: finish notebooks 1 and 2 first.')
    lock=read_json(path);runs,_,missing,invalid=inspect_development(campaign,package)
    require(not missing and not invalid,'Development results incomplete/invalid; final training blocked. Missing/incomplete: '+str(missing)+'; invalid: '+str(invalid))
    required=[f'{name}/seed_{seed}' for name in package.plan['models'] for seed in package.plan['seeds']]
    selection=select_common_epoch({k:v['result'] for k,v in runs.items()},required);selection.update(invalid_runs=invalid,missing_or_incomplete_runs=missing)
    expected=expected_final_lock(package,runs,selection)
    require(lock==expected,'Final lock differs from the complete development evidence / feature / protocol / architecture / logic version.')
    require(not lock['official_val_metrics_used'] and not lock['warm_start'],'Official-val/warm-start decision in final lock is forbidden.')
    return lock

def summarize_final(campaign,package,lock,status):
    per_seed=[];grouped=[]
    for name in package.plan['models']:
        for seed in package.plan['seeds']:
            key=f'{name}/seed_{seed}';row={'model_id':name,'model':package.plan['models'][name]['label'],'seed':seed,'status':status.get(key,{}).get('status','not_started'),'reason':status.get(key,{}).get('reason','')}
            for k in KS:row[f'Overall R@{k}']=None
            row['Mean']=None
            folder=run_directory(campaign,'final',name,seed)
            if (folder/'completion.json').is_file():
                try:
                    cfg=training_config(package,name,seed,'final',lock['selected_epoch'],lock);metric=verify_completion(folder,fingerprint(cfg),package,'final')
                    row.update(status='evaluated',reason='',Mean=metric['Mean'],**{f'Overall R@{k}':metric['overall'][f'R@{k}'] for k in KS})
                except Exception as exc:row.update(status='invalid',reason=redact(exc))
            per_seed.append(row)
        members=[r for r in per_seed if r['model_id']==name and r['status']=='evaluated'];complete=len(members)==len(package.plan['seeds'])
        group={'model_id':name,'Model':package.plan['models'][name]['label'],'seeds_evaluated':len(members),'seeds_expected':len(package.plan['seeds']),'complete':complete,'std_ddof':1}
        for label in ('Overall R@1','Overall R@5','Overall R@10','Overall R@50','Mean'):
            values=[r[label] for r in members];group[label+'_mean']=float(np.mean(values)) if complete else None;group[label+'_std']=float(np.std(values,ddof=1)) if complete and len(values)>1 else None
        grouped.append(group)
    folder=Path(campaign)/'final';folder.mkdir(exist_ok=True);write_csv(folder/'benchmark_per_seed.csv',pd.DataFrame(per_seed));write_csv(folder/'benchmark_summary.csv',pd.DataFrame(grouped))
    completed=all(r['status']=='evaluated' for r in per_seed)
    write_json(folder/'execution_status.json',{'phase':'final','feature_fingerprint':package.key,'lock_sha256':fingerprint(lock),'end_to_end_completed':completed,'runs':per_seed,'training_seed_aggregation':'mean and sample standard deviation ddof=1 over ALL configured seeds; no best official-val seed selection','paired_bootstrap_included':False})
    atomic_bytes(folder/'summary.md',(f"# Official validation\n- Fully evaluated: {sum(r['status']=='evaluated' for r in per_seed)}/{len(per_seed)} runs.\n- E*: {lock['selected_epoch']}; exact cosine, per-category macro, percent.\n- Mean ± sample std uses all configured seeds. Missing metrics stay blank; no best-seed selection or SOTA claim.\n").encode())
    return pd.DataFrame(grouped)


In [ ]:
def evaluator_fixture_checks():
    # Reference stays in the gallery. Exact equal scores use manifest index, including target ties.
    q=np.array([[1.,0.],[0.,1.]],np.float32);g=np.array([[1.,0.],[1.,0.],[0.,1.]],np.float32)
    ranks,top,_,ties=exact_rank(q,g,np.array([1,2]))
    require(ranks.tolist()==[2,1] and top[0].tolist()==[0,1,2] and ties.tolist()==[2,1],'Hand-computed rank/tie/reference test failed.')
    metrics=recall_metrics([1,2,6,11,51]);require(metrics=={'R@1':20.,'R@5':40.,'R@10':60.,'R@50':80.},'Hand recall failed.')
    cats={'dress':recall_metrics([1]),'shirt':recall_metrics([11,51]),'toptee':recall_metrics([6,6,6])}
    result=aggregate_categories(cats);require(abs(result['overall']['R@10']-200/3)<1e-10 and abs(result['Mean']-75.)<1e-10,'Unweighted macro/Mean test failed.')
    return {'hand_ranking_and_ties':True,'reference_retained':True,'recall_percent':True,'unweighted_macro_and_Mean':True}

def model_preflight(package,real_features=True):
    before=state_digest(capture_rng());report={'input_source':'actual prepared shards' if real_features else 'explicit local test fixture','models':{},'evaluator':evaluator_fixture_checks()}
    with preserve_rng():
        n=min(3,len(package.frames['development_train']));require(n>=2,'Need >=2 training rows for preflight.')
        batch=next(iter(make_loader(package.frames['development_train'].iloc[:n],package,n,False,torch.Generator().manual_seed(991),False)))
        r=batch['reference_hidden'].to(DEVICE);t=batch['text_hidden'].to(DEVICE);mask=batch['text_mask'].to(DEVICE);target=batch['target'].to(DEVICE);ids=batch['target_index'].to(DEVICE)
        expected=((n,50,768),(n,77,512),(n,77),(n,512));require(tuple(r.shape)==expected[0] and tuple(t.shape)==expected[1] and tuple(mask.shape)==expected[2] and tuple(target.shape)==expected[3],'Real feature batch shape mismatch.')
        require(all(torch.isfinite(x).all().item() for x in (r,t,target)) and (mask.sum(1)>0).all().item(),'Invalid preflight feature batch.')
        common_state=None;after_initialization=None;baseline_output=None
        for name in package.plan['models']:
            set_seed(package.plan['seeds'][0]);model=build_model(package.plan,name).float();common={k:v for k,v in model.state_dict().items() if not k.startswith(('mr.','sr.'))};rng=torch.get_rng_state().clone()
            if common_state is None:common_state=cpu_clone(common);after_initialization=rng
            else:require(all(torch.equal(common_state[k],v) for k,v in common.items()) and torch.equal(rng,after_initialization),'Shared XAA initialization/RNG unfair across variants.')
            model=model.to(DEVICE);model.eval()
            with torch.no_grad():
                out=query_forward(model,r,t,mask);single=query_forward(model,r[:1],t[:1],mask[:1])
                require(tuple(out.shape)==(n,512) and torch.isfinite(out).all().item() and torch.allclose(out.norm(dim=1),torch.ones(n,device=DEVICE),atol=1e-5),'Output shape/finite/norm failed.')
                require(torch.allclose(single,out[:1],atol=2e-5,rtol=2e-5),'Batch size 1 equivalence failed.')
                if baseline_output is None:baseline_output=out.clone()
                else:require(torch.allclose(out,baseline_output,atol=1e-6,rtol=1e-5),'Identity-initialized DVR changed XAA output.')
                changed=t.clone();changed[mask==0]=123.456
                require(torch.allclose(out,query_forward(model,r,changed,mask),atol=1e-5,rtol=1e-5),'Masked padding influences pooled output.')
            model.train();optimizer,_,_=optimizer_scheduler(model,package.h,1,1);optimizer.zero_grad(set_to_none=True)
            loss=multi_positive_infonce(query_forward(model,r,t,mask),target,ids,model.logit_scale);loss.backward()
            gradients={key:p.grad for key,p in model.named_parameters()};require(all(v is not None and torch.isfinite(v).all().item() for v in gradients.values()),'Missing/nonfinite head gradients: '+name)
            torch.nn.utils.clip_grad_norm_(model.parameters(),package.h['gradient_clip_norm'],error_if_nonfinite=True);optimizer.step()
            require(all(torch.isfinite(p).all().item() for p in model.parameters()),'Nonfinite preflight update.')
            report['models'][name]={'forward_backward_and_optimizer_step':True,'batch_one_matches_batch':True,'masked_pooling':True,'shared_initialization_and_rng':True,'finite_loss':float(loss.detach().cpu()),'gradient_tensors':len(gradients),'temporary_model_only':True}
            del optimizer,model
        set_seed(package.plan['seeds'][0]);base=build_model(package.plan,'xaa_noft').to(DEVICE).eval();set_seed(package.plan['seeds'][0]);off=XAADVR(use_mr=False,use_sr=False).to(DEVICE).eval()
        with torch.no_grad():require(torch.allclose(query_forward(base,r,t,mask),query_forward(off,r,t,mask),atol=1e-6,rtol=1e-5),'DVR-off differs from XAA.')
        # Duplicate positives use the ORIGINAL mean-log-probability loss, not a substituted positive-logsumexp.
        comp=F.normalize(torch.tensor([[1.,0.],[1.,0.],[0.,1.]],device=DEVICE),dim=-1);same=torch.tensor([7,7,9],device=DEVICE);scale=torch.tensor(0.,device=DEVICE)
        logits=comp@comp.T;positive=same[:,None]==same[None,:]
        manual=.5*((-F.log_softmax(logits,1)*positive).sum(1)/positive.sum(1)).mean()+.5*((-F.log_softmax(logits.T,1)*positive.T).sum(1)/positive.T.sum(1)).mean()
        require(torch.allclose(multi_positive_infonce(comp,comp,same,scale),manual,atol=1e-7),'Duplicate target loss failed.')
        all_same=torch.tensor([7,7,7],device=DEVICE);actual=multi_positive_infonce(torch.ones(3,2,device=DEVICE),torch.ones(3,2,device=DEVICE),all_same,scale)
        require(abs(float(actual)-math.log(3))<1e-6,'Original all-positive mean-log-probability loss changed.')
        report.update(dvr_off_equivalent=True,duplicate_positive_loss=True,all_positive_loss_is_log_batch_size=True)
    require(state_digest(capture_rng())==before,'Smoke test changed main RNG state.')
    report.update(rng_restored=True,passed=True,environment=execution_environment());return report


In [ ]:
def colab_guard():
    require('google.colab' in sys.modules,'Full feature-package training is intended for Colab. Local validation uses explicit temporary fixtures only.')

def load_and_preflight():
    colab_guard()
    global DEVICE
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.set_float32_matmul_precision('highest')
    if torch.cuda.is_available():torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    print('Device:',DEVICE,'| feature-only FP32; no encoder import, download, or training.')
    preflight_clock=time.perf_counter();stage=CFG['phase'];report_dir=campaign_path()/'preflight';report_dir.mkdir(parents=True,exist_ok=True)
    # A failed preflight never starts any training/evaluation and preserves full redacted traceback.
    try:
        metadata_clock=time.perf_counter()
        package=PreparedPackage(CFG['preparation_root'],CFG['local_feature_cache'],PACKAGE_ANCHORS,full_check=False)
        metadata_s=time.perf_counter()-metadata_clock;check_baseline_identity(package);legacy_inventory=inventory_legacy_training(package)
        if stage=='final':
            require((campaign_path()/'campaign_identity.json').is_file(),'No completed development campaign. Run notebooks 1 and 2 first.')
            lock=load_final_lock(campaign_path(),package)
        else:lock=None
        print('Verifying all committed NPY shards and making a local memmap mirror (~6.4 GB); no image/dataset download.')
        cache_clock=time.perf_counter();integrity=package.verify_all();copy_cache_s=time.perf_counter()-cache_clock;campaign=initialize_campaign(package)
        smoke_clock=time.perf_counter();preflight=model_preflight(package,real_features=True);smoke_s=time.perf_counter()-smoke_clock
        profile_clock=time.perf_counter();profile=short_profile(package);profile_s=time.perf_counter()-profile_clock
        timing={'metadata_preflight_s':metadata_s,'full_checksum_copy_cache_s':copy_cache_s,'model_preflight_s':smoke_s,'short_profile_s':profile_s,'total_preflight_s':time.perf_counter()-preflight_clock}
        write_json(report_dir/(stage+'_timing.json'),timing)
        write_json(report_dir/(stage+'_profile.json'),profile)
        write_json(report_dir/(stage+'_preflight.json'),{'passed':True,'environment':execution_environment(),'package_integrity':integrity,'model_and_metric_tests':preflight,'preparation':package.binding,'bootstrap':BOOTSTRAP_REPORT,'timing':timing,'short_profile':profile,'legacy_training_inventory':legacy_inventory,'baseline_campaign_read_only':str(baseline_campaign())})
        print('Preflight passed for all four configurations. Training plan seeds:',package.plan['seeds'])
        return package,campaign,lock
    except Exception:
        write_json(report_dir/(stage+'_failure_'+time.strftime('%Y%m%dT%H%M%S')+'_'+uuid.uuid4().hex[:8]+'.json'),{'passed':False,'phase':stage,'traceback':redact(traceback.format_exc()),'bootstrap':BOOTSTRAP_REPORT})
        raise

def run_selected(package,campaign,lock=None):
    colab_guard();phase=CFG['phase'];names=CFG['models_to_run'];seeds=package.plan['seeds'] if CFG['seeds_to_run'] is None else CFG['seeds_to_run']
    require(names and len(set(names))==len(names) and set(names)<=set(package.plan['models']),'Invalid models_to_run.')
    allowed={'development_dvr':{'xaa_mr','xaa_sr','xaa_full_dvr'},'final':set(package.plan['models'])}
    require(set(names)<=allowed[phase],'Model configuration outside this notebook phase.')
    require(seeds and len(set(seeds))==len(seeds) and set(seeds)<=set(package.plan['seeds']),'Selected seeds must be a nonempty subset of the locked plan.')
    stage='final' if phase=='final' else 'development';statuses={}
    if stage=='final':require(load_final_lock(campaign,package)==lock,'Final lock changed after preflight.')
    for name in names:
        for seed in seeds:
            key=f'{name}/seed_{seed}';folder=run_directory(campaign,stage,name,seed)
            try:
                train_run(package,campaign,name,seed,stage,lock)
                statuses[key]={'status':'evaluated' if stage=='final' else 'development_complete'}
            except Exception as exc:
                # Separate attempt files preserve the last committed epoch and other runs.
                failure={'status':'failed','reason':redact(exc),'traceback':redact(traceback.format_exc()),'model':name,'seed':seed,'phase':stage,'preparation':package.binding}
                folder.mkdir(parents=True,exist_ok=True);write_json(folder/('failure_attempt_'+time.strftime('%Y%m%dT%H%M%S')+'_'+uuid.uuid4().hex[:8]+'.json'),failure)
                statuses[key]={'status':'failed','reason':failure['reason']};print('Run failed:',key,redact(exc))
                if torch.cuda.is_available():torch.cuda.empty_cache()
    if stage=='development':
        summary,selection=refresh_development_summary(campaign,package)
        write_json(Path(campaign)/stage/'execution_status.json',{'phase':phase,'attempted':statuses,'all_development_complete':bool(selection['ready']),'final_lock_ready':bool(selection['ready']),'official_val_used':False,'metric_for_unexecuted_runs':None})
        print(summary.to_string(index=False));print('Eligible E* epochs:',selection['eligible_epochs'],'| final lock ready:',selection['ready'])
    else:
        summary=summarize_final(campaign,package,lock,statuses);print(summary.to_string(index=False))
    print('Artifacts:',Path(campaign)/stage)
    if any(row['status']=='failed' for row in statuses.values()):raise RuntimeError('One or more runs failed; see per-run failure_attempt JSON and rerun to resume from the last saved epoch.')
    return summary


Preflight kiểm tra toàn bộ nguồn và cache, rồi chỉ dùng metadata RAM/feature local trong vòng batch. Cache local mất/thay đổi sẽ dừng; chạy lại cell preflight để kiểm tra và sửa bản local. `profile_batches=2` chỉ chạy head tạm, phục hồi RNG; đặt 0 nếu muốn bỏ profiling. Timing ở JSON/CSV: preflight/cache, data loading, update, validation, save và tổng epoch; CUDA event chỉ đồng bộ tại ranh giới phase.


In [ ]:
package, campaign, final_lock = load_and_preflight()


In [ ]:
result_summary = run_selected(package, campaign, final_lock)


Output mới: `taskb_training/FashionIQ/heads_v1_20261103_ioopt_v1/{development,final}`. Baseline và artifact cũ giữ nguyên. Resume nghiêm ngặt theo config/implementation/runtime; không hứa bit-exact giữa thiết bị. Chưa đo tăng tốc trên Colab; mọi output trong notebook này được để trống.
